In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Connect to the Azure ML workspace
ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

print("Workspace name:", ml_client.workspace_name)
print("Subscription id:", ml_client.subscription_id)
print("Resource group:", ml_client.resource_group_name)

In [ ]:
from azure.ai.ml.entities import AmlCompute

cpu_cluster_name = "lab-cluster-compute-notebook"

# Check whether the compute cluster already exists
try:
    compute_target = ml_client.compute.get(cpu_cluster_name)
    print(f"Found existing cluster: {cpu_cluster_name}")

except Exception:
    print("Creating a new compute cluster...")

    compute_target = AmlCompute(
        name=cpu_cluster_name,
        type="amlcompute",
        size="Standard_D2_v2",
        min_instances=0,
        max_instances=4,
        idle_time_before_scale_down=120,
    )

    compute_target = ml_client.compute.begin_create_or_update(
        compute_target
    ).result()

    print(f"Created compute cluster: {cpu_cluster_name}")

print(f"Compute target: {compute_target.name}")
print(f"VM size: {compute_target.size}")
print(f"Max nodes: {compute_target.max_instances}")

In [ ]:
from azure.ai.ml.entities import Environment


conda_yaml = """
name: udacity-sklearn-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - pip
  - pip:
      - scikit-learn
      - pandas
      - numpy
      - mlflow<3
      - azureml-mlflow
"""

with open("conda.yml", "w") as f:
    f.write(conda_yaml)

print("conda.yml created.")



sklearn_env = Environment(
    name="udacity-sklearn-env",
    description="Scikit-learn environment for Udacity project",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml"
)

sklearn_env = ml_client.environments.create_or_update(
    sklearn_env
)

print("Environment created:")
print("  Name:", sklearn_env.name)
print("  Version:", sklearn_env.version)

In [ ]:
from azure.ai.ml import command
from azure.ai.ml.sweep import Choice, BanditPolicy

job = command(
    code="./",
    command=(
        "python train.py "
        "--C ${{inputs.C}} "
        "--max_iter ${{inputs.max_iter}}"
    ),
    environment=f"{sklearn_env.name}:{sklearn_env.version}",
    compute=cpu_cluster_name,
    inputs={
        "C": 1.0,
        "max_iter": 100
    }
)

print("Base command job created.")


sweep_job = job(
    C=Choice(values=[
        0.001,
        0.01,
        0.1,
        1,
        10,
        20,
        50,
        100,
        200,
        500,
        1000
    ]),
    max_iter=Choice(values=[
        50,
        100,
        200,
        300
    ])
).sweep(
    compute=cpu_cluster_name,
    sampling_algorithm="random",
    primary_metric="Accuracy",
    goal="maximize"
)

# Early termination
sweep_job.early_termination = BanditPolicy(
    evaluation_interval=2,
    slack_factor=0.1
)

# Limits
sweep_job.set_limits(
    max_total_trials=16,
    max_concurrent_trials=4,
    timeout=3600
)

# Naming
sweep_job.display_name = "udacity-hyperparameter-tuning"
sweep_job.experiment_name = "udacity-project"

print("Sweep job configured.")

returned_sweep_job = ml_client.jobs.create_or_update(
    sweep_job
)

print("========================================")
print("Sweep job submitted successfully!")
print("========================================")
print("Job name:", returned_sweep_job.name)
print("Studio URL:", returned_sweep_job.studio_url)

In [ ]:
ml_client.jobs.stream(
    returned_sweep_job.name
)

print(returned_sweep_job.studio_url)

# Get the completed sweep
sweep_job = ml_client.jobs.get(
    returned_sweep_job.name
)

print("Sweep status:", sweep_job.status)

# Get child trial jobs
children = list(
    ml_client.jobs.list(
        parent_job_name=sweep_job.name
    )
)

# Keep completed trials
completed_children = [
    child
    for child in children
    if child.status == "Completed"
]

# Sort by Accuracy
completed_children.sort(
    key=lambda child: float(
        child.properties.get(
            "mlflow.metrics.Accuracy",
            0
        )
    ),
    reverse=True
)

print("Trials sorted by Accuracy:\n")

for child in completed_children:
    print(
        f"Run: {child.name}, "
        f"Accuracy: "
        f"{child.properties.get('mlflow.metrics.Accuracy')}"
    )

if not completed_children:
    raise RuntimeError(
        "No completed trials were found."
    )

best_run = completed_children[0]

print("\n==============================")
print("BEST RUN")
print("==============================")

print("Run ID:", best_run.name)
print(
    "Accuracy:",
    best_run.properties.get(
        "mlflow.metrics.Accuracy"
    )
)

In [ ]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###

In [ ]:
from train import clean_data

# Use the clean_data function to clean your data.
x, y = clean_data(### YOUR DATA OBJECT HERE ###)

In [ ]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task=,
    primary_metric=,
    training_data=,
    label_column_name=,
    n_cross_validations=)

In [2]:
# Submit your automl run

### YOUR CODE HERE ###

In [ ]:
# Retrieve and save your best automl model.

### YOUR CODE HERE ###